# Path A, P12 — CIs on the rule's weights (Oct 27, 2026)

Pre-registered in the Path A Workbook, Ch. A8.2. No choice below was changed. Training data only: no G-val row is used (they are dropped right after loading), no Ring or CT.

- **Model:** M2' of P11, R0 structure: const + lab + A, |s| (`abs_s`), `at_end`, e (`time_at_end_log1p`), r (`time_since_rev_log1p`); outcome `y_rev`; all training cells (Hierarchy, Full, Star), G-train rows; `fit.fit` (statsmodels Logit, unpenalized, raw features).
- **Bootstrap:** 1,000 resamples, seed 573. G-train groups (log files) are drawn with replacement within each dataset (EL, CL, CC), as many as in the original; all bins of each drawn group are kept (a group drawn twice counts twice); M2' is refit on each resample. Failed or unconverged refits are counted; the CIs use the converged refits.
- **Per weight:** point estimate and model SE (P11, `p11_coefs.csv`), bootstrap SE (SD of the refit betas), ratio bootstrap SE / model SE, 95% percentile CI.
- **Derived:** exp(beta) per weight; crossover time t* = exp(beta_at_end / −beta_e) − 1 (seconds at the end). Bootstrap CIs: percentiles of the quantity over the refits.
- **Check:** the refit on the original G-train rows reproduces P11's M2' betas exactly.

Saved: `results/tables/p12_beta_ci.csv`.

In [1]:
import warnings

import numpy as np
import pandas as pd

from slider import config
from slider import fit

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

FEATS = ["A", "abs_s", "at_end", "time_at_end_log1p", "time_since_rev_log1p"]     # M2' (P11)
N_BOOT = 1000

ht = pd.read_parquet(config.HAZARD_TABLE)
ht = ht[ht["topology"].isin(config.TRAIN_TOPOLOGIES)]
train = ht[ht["split"] == "train"].reset_index(drop=True)                     # G-train only
del ht
strata = {d: np.array(sorted(train.loc[train["dataset"] == d, "group_id"].unique())) for d in ["EL", "CL", "CC"]}
print(len(train), "G-train bins,", int(train["y_rev"].sum()), "reversals; groups per dataset:",
      {d: len(g) for d, g in strata.items()})

336073 G-train bins, 6073 reversals; groups per dataset: {'EL': 11, 'CL': 11, 'CC': 17}


## Check: refit on the original G-train rows = P11's M2'

In [2]:
p11 = pd.read_csv(config.P11_COEFS, float_precision="round_trip")      # round-trip parsing: exact floats
p11 = p11[p11["model"] == "M2'"].set_index("term")
m0 = fit.fit(train, FEATS, False)
terms = list(m0.params.index)
same = pd.DataFrame({"P11 beta": p11.loc[terms, "beta"], "refit beta": m0.params, "P11 se": p11.loc[terms, "se"], "refit se": m0.bse})
same["beta equal"] = same["P11 beta"] == same["refit beta"]
same["se equal"] = same["P11 se"] == same["refit se"]
print("refit reproduces P11's M2' betas exactly:", bool(same["beta equal"].all()),
      "| max |diff| =", float((same["P11 beta"] - same["refit beta"]).abs().max()))
print("converged:", m0.mle_retvals["converged"])
same

refit reproduces P11's M2' betas exactly: True | max |diff| = 0.0
converged: True


,P11 beta,refit beta,P11 se,refit se,beta equal,se equal
const,-3.194120,-3.194120,0.053751,0.053751,True,True
lab,0.057821,0.057821,0.027061,0.027061,True,True
A,-2.107405,-2.107405,0.047707,0.047707,True,True
abs_s,1.047643,1.047643,0.087873,0.087873,True,True
at_end,0.444558,0.444558,0.061493,0.061493,True,True
time_at_end_log1p,-0.377448,-0.377448,0.024083,0.024083,True,True
time_since_rev_log1p,-0.209264,-0.209264,0.024790,0.024790,True,True


## Bootstrap: 1,000 stratified group resamples of G-train, M2' refit on each

In [3]:
rows = train.groupby("group_id").indices                  # positional rows of each group
rng = np.random.default_rng(config.SEED)
betas, log = [], []
for b in range(N_BOOT):
    drawn = np.concatenate([g[rng.integers(0, len(g), len(g))] for d, g in strata.items()])   # within each dataset
    idx = np.concatenate([rows[g] for g in drawn])
    entry = {"resample": b, "bins": len(idx), "reversals": int(train["y_rev"].to_numpy()[idx].sum())}
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            m = fit.fit(train.iloc[idx].reset_index(drop=True), FEATS, False)
        ok = bool(m.mle_retvals["converged"] and np.isfinite(m.params).all())
        entry.update(status="converged" if ok else "unconverged", warnings=len(caught))
        if ok:
            betas.append(m.params.rename(b))
    except Exception as e:
        entry.update(status=f"failed: {type(e).__name__}: {e}", warnings=np.nan)
    log.append(entry)
log = pd.DataFrame(log)
B = pd.DataFrame(betas)
print("refits:", log["status"].value_counts().to_dict(), "| refits with any warning:", int((log["warnings"] > 0).sum()))
print("strata order of the draws:", list(strata))
log[["bins", "reversals"]].describe().round(1).T

refits: {'converged': 1000} | refits with any warning: 0
strata order of the draws: ['EL', 'CL', 'CC']


,count,mean,std,min,25%,50%,75%,max
bins,1000.0,336901.9,23401.2,257893.0,320751.5,336954.0,352584.8,403232.0
reversals,1000.0,6077.6,481.9,4663.0,5747.8,6061.5,6406.0,7501.0


## Weights: P11 estimate, model SE, bootstrap SE, ratio, 95% percentile CI; exp(beta); t*

In [4]:
def t_star(at_end, e):
    """Crossover time (s at the end): exp(beta_at_end / -beta_e) - 1."""
    return np.exp(at_end / -e) - 1


est, model_se = p11.loc[terms, "beta"], p11.loc[terms, "se"]
lo, hi = np.percentile(B[terms], [2.5, 97.5], axis=0)
elo, ehi = np.percentile(np.exp(B[terms]), [2.5, 97.5], axis=0)
table = pd.DataFrame({"quantity": terms, "estimate": est.to_numpy(), "model_se": model_se.to_numpy(),
                      "boot_se": B[terms].std(ddof=1).to_numpy(), "ci_low": lo, "ci_high": hi,
                      "exp_estimate": np.exp(est).to_numpy(), "exp_ci_low": elo, "exp_ci_high": ehi})
table.insert(4, "ratio", table["boot_se"] / table["model_se"])

ts = t_star(B["at_end"], B["time_at_end_log1p"])
print("refits with beta_e >= 0 (t* not defined as a crossover):", int((B["time_at_end_log1p"] >= 0).sum()))
table = pd.concat([table, pd.DataFrame([{"quantity": "t_star_s", "estimate": t_star(est["at_end"], est["time_at_end_log1p"]),
                                         "boot_se": ts.std(ddof=1), "ci_low": np.percentile(ts, 2.5),
                                         "ci_high": np.percentile(ts, 97.5)}])], ignore_index=True)
table["n_boot"] = len(B)
table.to_csv(config.P12_BETA_CI, index=False)
table.round(4)

refits with beta_e >= 0 (t* not defined as a crossover): 0


,quantity,estimate,model_se,boot_se,ratio,ci_low,ci_high,exp_estimate,exp_ci_low,exp_ci_high,n_boot
0,const,-3.1941,0.0538,0.0929,1.7286,-3.3871,-3.0274,0.0410,0.0338,0.0484,1000
1,lab,0.0578,0.0271,0.0866,3.2001,-0.1240,0.2136,1.0595,0.8834,1.2381,1000
2,A,-2.1074,0.0477,0.1673,3.5068,-2.4500,-1.7725,0.1216,0.0863,0.1699,1000
3,abs_s,1.0476,0.0879,0.1145,1.3027,0.8287,1.2760,2.8509,2.2904,3.5822,1000
4,at_end,0.4446,0.0615,0.0746,1.2138,0.2966,0.5899,1.5598,1.3453,1.8038,1000
5,time_at_end_log1p,-0.3774,0.0241,0.0542,2.2503,-0.4901,-0.2792,0.6856,0.6126,0.7564,1000
6,time_since_rev_log1p,-0.2093,0.0248,0.0757,3.0544,-0.3535,-0.0514,0.8112,0.7023,0.9499,1000
7,t_star_s,2.2472,NaN,0.7995,NaN,1.1135,4.3071,NaN,NaN,NaN,1000
